# Lab 004 — Returns & the Stylized Facts

**Lesson:** [`0004-returns-stylized-facts.html`](../lessons/0004-returns-stylized-facts.html) · **Cheat sheet:** [`stylized-facts.html`](../reference/stylized-facts.html)

**The one skill:** compute log returns and *verify the stylized facts* in data — fat tails, absent linear autocorrelation, volatility clustering — and see what a Gaussian series is missing.

**Exit criteria:** the EXIT TICKET cell prints cleanly (all CHECK cells pass).

**How this notebook works**

| Cell tag | You do |
|----------|--------|
| **PROVIDED** | Run it. Imports, data, helpers. |
| **TODO** | Fill the `____` blanks. This is where the learning is. |
| **CHECK** | Run it — immediate assertions. Don't edit. |
| **EXIT TICKET** | Final deliverable. Prints your findings. |

**Environment:** Python 3 + `numpy`, `pandas`, `scipy`. See [`labs/README.md`](./README.md).

## Concept recap (read before coding)

**Log return:** $r_t = \ln(P_t / P_{t-1})$. Log returns are additive across time and symmetric around 0.

**The three facts you'll measure:**
1. **Fat tails** — excess kurtosis $\gg 0$ (a normal has excess kurtosis 0). Extreme moves are far more common than Gaussian predicts.
2. **No linear autocorrelation** — $\mathrm{corr}(r_t, r_{t+1}) \approx 0$. You can't predict direction from the last return alone.
3. **Volatility clustering** — $\mathrm{corr}(|r_t|, |r_{t+1}|) > 0$. Big moves follow big moves, so *volatility* is forecastable even though direction isn't.

**Toy worked example** (not the lab's answer). For returns `[0.01, -0.02, 0.015, -0.03]`, the sign flips every step → raw lag-1 autocorrelation is *negative*, while the magnitudes `[.01,.02,.015,.03]` trend up → |r| autocorrelation is *positive*. That gap between raw and absolute autocorrelation is exactly volatility clustering.

In [12]:
# PROVIDED — imports and data. Tries real data; falls back to a realistic
# simulation with fat tails + volatility clustering so the lab always runs offline.
import numpy as np
import pandas as pd
from scipy import stats

def load_prices(ticker="SPY", n=2500, seed=7):
    """Return a pandas Series of daily close prices."""
    try:
        import yfinance as yf  # optional; only if you have it + network
        px = yf.download(ticker, period="10y", progress=False)["Close"].dropna()
        if len(px) > 100:
            print(f"Loaded {len(px)} real daily closes for {ticker}.")
            return px
    except Exception as e:
        print(f"(Real data unavailable: {e}) — using a simulated series.")
    # Fallback: GARCH-like vol clustering + Student-t (fat-tailed) innovations.
    rng = np.random.default_rng(seed)
    omega, alpha, beta = 1e-6, 0.08, 0.90
    var = np.empty(n); ret = np.empty(n)
    var[0] = omega / (1 - alpha - beta)
    for t in range(1, n):
        var[t] = omega + alpha * ret[t-1]**2 + beta * var[t-1]
        ret[t] = np.sqrt(var[t]) * rng.standard_t(df=4) / np.sqrt(4/2)
    price = 100 * np.exp(np.cumsum(ret))
    print(f"Simulated {n} daily closes (fat tails + vol clustering).")
    return pd.Series(price, name="close")

px = load_prices()
px.head()

Loaded 2512 real daily closes for SPY.


Ticker,SPY
Date,
2016-07-14,184.093842
2016-07-15,183.846893
2016-07-18,184.340897
2016-07-19,184.153488
2016-07-20,184.920151


### Task 1 — Log returns
**Goal:** turn the price series into a clean log-return series `r`.

**Why it matters:** every downstream fact is a property of returns, not prices. Getting the transform right (and dropping the first NaN) is the foundation.

**Hint boundary:** use `np.log` on the ratio of consecutive prices, then drop the NaN. Do *not* use simple percentage returns here.

In [13]:
# TODO — compute log returns of px, drop the leading NaN, name the result `r`.
r = np.log(px / px.shift(1))
r = r.dropna().SPY
r.describe()

count    2511.000000
mean        0.000559
std         0.011320
min        -0.115887
25%        -0.003587
50%         0.000718
75%         0.005931
max         0.099863
Name: SPY, dtype: float64

In [14]:
# CHECK — do not edit
assert isinstance(r, pd.Series), "r must be a pandas Series"
assert r.isna().sum() == 0, "drop the NaN(s)"
assert abs(r.mean()) < 0.01, "daily log returns should have a near-zero mean"
print("Task 1 OK — %d log returns" % len(r))

Task 1 OK — 2511 log returns


### Task 2 — Fat tails
**Goal:** compute the **excess kurtosis** of `r` and compare it to a normal (0).

**Why it matters:** if kurtosis is large, any risk model built on the normal distribution will undercount crash risk.

**Hint boundary:** `scipy.stats.kurtosis` returns *excess* kurtosis by default (Fisher). Store it in `excess_kurt`.

In [16]:
# TODO — excess kurtosis of r (scipy Fisher definition).
excess_kurt = stats.kurtosis(r)
print(f"excess kurtosis = {excess_kurt:.2f}  (normal = 0)")

excess kurtosis = 15.39  (normal = 0)


In [17]:
# CHECK — do not edit
assert excess_kurt > 1.0, "real/simulated returns should show clear excess kurtosis (fat tails)"
print("Task 2 OK — fat tails confirmed")

Task 2 OK — fat tails confirmed


### Task 3 — The autocorrelation gap (volatility clustering)
**Goal:** compute lag-1 autocorrelation of **raw** returns vs **absolute** returns.

**Why it matters:** raw ≈ 0 (direction unpredictable) but |r| > 0 (volatility predictable) is the single most important asymmetry a QR exploits — model vol, not direction, from the past.

**Hint boundary:** `Series.autocorr(lag=1)`. Apply it to `r` and to `r.abs()`.

In [18]:
# TODO — lag-1 autocorrelation of raw and absolute returns.
ac_raw = r.autocorr(lag=1)
ac_abs = r.abs().autocorr(lag=1)
print(f"autocorr(r, 1)   = {ac_raw:+.3f}")
print(f"autocorr(|r|, 1) = {ac_abs:+.3f}")

autocorr(r, 1)   = -0.133
autocorr(|r|, 1) = +0.363


In [19]:
# CHECK — do not edit
assert abs(ac_raw) < 0.15, "raw-return autocorrelation should be small"
assert ac_abs > ac_raw, "|return| autocorrelation should exceed raw — that's volatility clustering"
print("Task 3 OK — volatility clustering confirmed")

Task 3 OK — volatility clustering confirmed


### Task 4 — What a Gaussian is missing
**Goal:** simulate an IID normal series with the *same mean and std* as `r`, and compare its excess kurtosis and |r| autocorrelation.

**Why it matters:** this makes concrete *why* the normal-IID assumption (behind naïve Sharpe/t-stat significance) is wrong for markets.

**Hint boundary:** draw `len(r)` samples from `np.random.default_rng(0).normal(mean, std, size)`; wrap in a `pd.Series`; reuse the same two measurements.

In [ ]:
# TODO — build a Gaussian comparison series `g` and measure it.
g = 
g = pd.Series(g)
g_kurt = stats.kurtosis(g)
g_ac_abs = g.abs().autocorr(lag=1)
print(f"Gaussian excess kurtosis = {g_kurt:+.2f}  (vs real {excess_kurt:+.2f})")
print(f"Gaussian autocorr(|g|,1) = {g_ac_abs:+.3f}  (vs real {ac_abs:+.3f})")

In [ ]:
# CHECK — do not edit
assert abs(g_kurt) < 1.0, "a Gaussian series should have ~0 excess kurtosis"
assert g_ac_abs < ac_abs, "the Gaussian series should NOT show volatility clustering"
print("Task 4 OK — the Gaussian is missing both fat tails and clustering")

In [ ]:
# EXIT TICKET — paste this output to your teacher.
print("=== Stylized-facts report ===")
print(f"n returns              : {len(r)}")
print(f"excess kurtosis (real) : {excess_kurt:+.2f}   [fat tails: >0]")
print(f"autocorr raw returns   : {ac_raw:+.3f}   [~0: no linear predictability]")
print(f"autocorr |returns|     : {ac_abs:+.3f}   [>raw: volatility clustering]")
print(f"Gaussian kurtosis      : {g_kurt:+.2f}   [~0: thin tails]")
print(f"Gaussian autocorr |g|  : {g_ac_abs:+.3f}   [~0: no clustering]")
print("\nOne-sentence takeaway (edit me):")
print("Real returns show fat tails and volatility clustering that an IID-normal series lacks,")
print("so any Sharpe/t-stat assuming normal-IID overstates significance.")

### Stretch (optional, ungraded)
- Plot `r.rolling(21).std()` — see the vol clusters with your eyes.
- Fit a Student-t to `r` with `stats.t.fit` and read off the degrees of freedom (lower ν = fatter tails).
- Compute the autocorrelation of `|r|` out to lag 20 and observe the slow decay (long memory in volatility → motivates HAR-RV in Unit 025).